<div style="padding: 28px; border-radius: 18px; background: linear-gradient(120deg, #07111f 0%, #0e2940 45%, #0f766e 100%); color: #f8fafc; margin-bottom: 14px;">
  <div style="font-size: 12px; text-transform: uppercase; opacity: 0.85;">NeMo Evaluator</div>
  <h1 style="margin: 8px 0 10px; font-weight: 800; line-height: 1.1;">Evaluate an Agent on a Real Benchmark, Publish to Intake</h1>
  <div style="font-size: 15px; opacity: 0.9;">Pull Terminal-Bench from Harbor Hub &rarr; run an agent in Docker &rarr; score &rarr; land it in Intake.</div>
</div>

## What this notebook does

**Harbor** runs each task in its own Docker container: the agent gets a workspace and an
instruction, a verifier script decides whether it succeeded, and Harbor stamps a reward.

**Harbor Hub** is its public registry — 80-odd benchmarks including SWE-bench Verified, GAIA,
GPQA-Diamond and Terminal-Bench.

**Intake** is the platform's telemetry store: trajectories and their
scores, queryable and grouped under an Evaluation.

This notebook connects all three:

![Harbor Hub to Intake: the registry supplies task folders, AgentEvaluator runs and scores them in
Docker, the AgentEvalResult carries trials and scores, and publish_to_intake writes the ATIF
trajectory and scores into Intake, where they are queryable.](pipeline.png)

We pull **Terminal-Bench 2.1** — Apache-2.0 licensed, 89 tasks that put an agent to work in a real
terminal — and run two of them against **codex**. What lands in Intake is that agent's actual
trajectory: the task it was given, its reasoning, the commands it ran, and what it produced.

## Prerequisites

- **Python >= 3.12** with the Harbor extra: `uv pip install "harbor>=0.16.1"`
- **Docker**, running — Harbor executes every task in a container, and Intake needs it for ClickHouse
- **A logged-in `codex`** (`codex login`, or `OPENAI_API_KEY`). The cell below picks up the
  `auth.json` that `codex login` writes.
  - Set `AGENT = "oracle"` to run with no credentials at all.
- **`make bootstrap-studio`**, only if you want the Studio links at the end to open anything — in a
  source checkout Studio has no UI until that bundle is built.

Intake is ClickHouse-backed and provisions a managed ClickHouse container itself, as long as
nothing has pointed it at an operator-owned one (`NMP_INTAKE_CLICKHOUSE_URL` unset and the resolved
URL still the default). On a stock checkout there is nothing to start separately.

In [ ]:
import os
import subprocess
import sys
import urllib.request
from importlib.util import find_spec
from pathlib import Path

import rich
from nemo_evaluator.intake import mapping as _mapping

# The two things every section below needs. Everything else is defined where it is used.
BASE_URL = os.environ.get("NMP_BASE_URL", "http://localhost:8080")
WORKSPACE = "default"

# Under $HOME rather than /tmp: Harbor bind-mounts the container's /logs back out to collect the
# verifier reward, and some macOS Docker backends only share paths under $HOME.
WORK_DIR = Path.home() / ".cache" / "harbor-to-intake-notebook"

assert find_spec("harbor") is not None, 'harbor is not installed: uv pip install "harbor>=0.16.1"'
assert subprocess.run(["docker", "info"], capture_output=True, check=False).returncode == 0, (
    "Docker is not available; Harbor runs every task in a container"
)
# Harbor renders trial progress with rich, which in a notebook emits an ipywidgets view *and* an
# HTML block — two copies of the same bar, in a panel that ignores the editor theme. Rich's
# Progress uses the global console, so reconfiguring it here settles how Harbor renders: plain
# stdout, and non-terminal so the spinner is not redrawn hundreds of times into the output.
rich.reconfigure(force_jupyter=False, force_terminal=False)

# A kernel keeps the modules it first imported, so edits to the installed evaluator do not reach a
# long-running session. Without this the run still succeeds and publishes a single-step trajectory,
# which looks like empty trace data in Studio rather than like a stale kernel.
assert hasattr(_mapping, "atif_steps_from_trial"), (
    "this kernel holds an older nemo_evaluator — restart the kernel to publish full trajectories"
)

print(f"platform : {BASE_URL}")
print(f"work dir : {WORK_DIR}")

<a id="platform"></a>
## 0. Start the platform

Rather than running `nemo services run` in another terminal, this starts it for you and waits until
it reports ready — about 30 seconds on a first run, since Intake provisions its ClickHouse
container.

A platform already answering on `BASE_URL` is **reused, never killed**, so this is safe to re-run
and safe alongside a platform you started elsewhere. Set `RESTART_PLATFORM = True` to replace one
this notebook started. Services include `studio` so the links in the last section have something to
open — though in a source checkout that also needs `make bootstrap-studio` to have built
`web/packages/studio/dist`. Without it Studio still starts and serves a "not built" notice.

The platform is stopped when this kernel shuts down.

In [ ]:
import atexit
import time

SERVICES = "auth,entities,intake,studio"
RESTART_PLATFORM = False
PLATFORM_LOG = WORK_DIR / "platform.log"


def platform_ready(timeout=2):
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health/ready", timeout=timeout) as response:
            return response.status == 200
    except OSError:
        return False


started = globals().get("_platform_proc")
if RESTART_PLATFORM and started is not None and started.poll() is None:
    started.terminate()
    started.wait(timeout=30)
    started = None

if platform_ready():
    print(f"platform : reusing the one already serving {BASE_URL}")
else:
    # cwd matters: the platform resolves its data directory relative to the repo checkout.
    repo_root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists()), Path.cwd())
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    _platform_proc = subprocess.Popen(
        [str(Path(sys.executable).parent / "nemo"), "services", "run", "--services", SERVICES],
        cwd=repo_root,
        stdout=PLATFORM_LOG.open("w"),
        stderr=subprocess.STDOUT,
        env={**os.environ, "NMP_BASE_URL": BASE_URL},
    )
    # Only ever terminate what this notebook started.
    atexit.register(_platform_proc.terminate)
    print(f"platform : starting {SERVICES}")
    print(f"           log: {PLATFORM_LOG}")
    for _ in range(60):
        if platform_ready():
            break
        if _platform_proc.poll() is not None:
            raise SystemExit(f"platform exited early - see {PLATFORM_LOG}")
        time.sleep(2)
    else:
        # SystemExit does not stop a Jupyter kernel, so atexit never runs here: stop the child
        # ourselves or it keeps holding the port after this cell fails.
        _platform_proc.terminate()
        _platform_proc.wait(timeout=30)
        raise SystemExit(f"platform not ready after 120s - see {PLATFORM_LOG}")

print(f"platform : ready at {BASE_URL}")

<a id="pull"></a>
## 1. Pull the tasks from Harbor Hub

`harbor download --export` lays a dataset out as `<output>/<dataset>/<task>/` — a directory whose
immediate subdirectories are task folders. That is exactly the shape the SDK's Harbor runtime
discovers, so nothing has to be reshaped after the download.

A task id is an `org/task` reference, so the Hub can serve them one at a time — the two below
come down in about 3 seconds each, rather than pulling all 89.

In [ ]:
TASKS = ["terminal-bench/git-leak-recovery", "terminal-bench/largest-eigenval"]

In [ ]:
from nemo_evaluator_sdk.agent_eval.runtimes.harbor_runtime import discover_harbor_tasks

# `--export` drops each task folder straight in here, which makes TASKS_DIR itself the
# "directory whose subdirectories are task folders" the runtime discovers.
TASKS_DIR = WORK_DIR / "tasks"
TASKS_DIR.mkdir(parents=True, exist_ok=True)
harbor_cli = Path(sys.executable).parent / "harbor"
for task in TASKS:
    folder = TASKS_DIR / task.rsplit("/", 1)[-1]
    if folder.is_dir() and any(folder.iterdir()):
        continue
    subprocess.run([str(harbor_cli), "download", task, "--export", "-o", str(TASKS_DIR)], check=True)

# Task ids come from each task.toml's [task] name, which need not match the folder name — so
# check the ids the runtime will actually see. Filtering to nothing otherwise fails later with
# "at least one task is required", which reads like a bug rather than a typo.
available = {task.id for task in discover_harbor_tasks(TASKS_DIR)}
unknown = sorted(set(TASKS) - available)
assert not unknown, f"unknown task id(s) {unknown}; {len(available)} available, e.g. {sorted(available)[:5]}"

print(f"tasks ready in {TASKS_DIR}")
print(f"running: {TASKS}")

<a id="anatomy"></a>
## 2. What a Harbor task actually is

Every task folder is self-describing — the instruction, the container it runs in, and the verifier
that decides the reward. Nothing here is NeMo-specific; this is the benchmark exactly as it is
published on the Hub.

In [ ]:
# A task's id is namespaced (`terminal-bench/git-leak-recovery`) while its folder is not, so take
# the directory discovery already resolved rather than rebuilding the path from the id.
sample = Path(next(t for t in discover_harbor_tasks(TASKS_DIR) if t.id == TASKS[0]).metadata["harbor_task_dir"])

print("files:", sorted(str(p.relative_to(sample)) for p in sample.rglob("*") if p.is_file()))
print("\n--- instruction.md ---")
print((sample / "instruction.md").read_text().strip()[:700])
print("\n--- tests/test.sh (the verifier) ---")
print((sample / "tests" / "test.sh").read_text().strip()[:500])

<a id="run"></a>
## 3. Run the evaluation

The evaluator needs two things: the **tasks** to score, and a **target** that produces trials for
them. The Harbor runtime supplies both — a taskset loaded from the task folders, and a runner that
executes each task in Docker and adapts Harbor's results back into trials.

`AgentEvaluator.run()` is what ties them together and scores the result. (`run_harbor_eval` is a
one-call wrapper around exactly this, if you would rather not assemble it yourself.)

Each task carries its own prebuilt image (~330 MB), pulled once and cached. Two tasks take a little
over a minute with `oracle`, including those pulls; with codex, add the agent's own turns.

In [ ]:
AGENT = "codex"
MODEL = "gpt-5.6-luna"

In [ ]:
from nemo_evaluator_sdk.agent_eval.evaluator import AgentEvaluator
from nemo_evaluator_sdk.agent_eval.runtimes.harbor_runtime import (
    HarborAgentTaskRunner,
    HarborRuntimeConfig,
    HarborTasksetLoader,
)
from nemo_evaluator_sdk.agent_eval.tasks import AgentEvalRunConfig

KEYLESS_AGENTS = {"oracle", "nop"}

# Sets the Codex credentials
codex_auth = Path.home() / ".codex" / "auth.json"
if AGENT == "codex" and not (os.environ.get("OPENAI_API_KEY") or os.environ.get("CODEX_FORCE_AUTH_JSON")):
    assert codex_auth.exists(), "the codex agent needs credentials: run `codex login`, or export OPENAI_API_KEY"
    os.environ["CODEX_FORCE_AUTH_JSON"] = "1"
    print(f"codex auth: using {codex_auth}")

result = await AgentEvaluator().run(
    tasks=HarborTasksetLoader(TASKS_DIR).load().tasks,
    target=HarborAgentTaskRunner(
        config=HarborRuntimeConfig(
            jobs_dir=WORK_DIR / "jobs",
            agent_name=AGENT,
            agent_model_name=None if AGENT in KEYLESS_AGENTS else MODEL,
            n_attempts=1,
            n_concurrent_trials=2,
        ),
        task_names=TASKS,
    ),
    config=AgentEvalRunConfig(),
)

print(f"run_id : {result.run_id}")
print(f"tasks  : {result.summary.task_count}   trials: {result.summary.trial_count}")
for aggregate in result.summary.scores.scores:
    print(f"mean   : {aggregate.name} = {aggregate.mean}")

In [ ]:
for score in result.scores:
    reward = score.outputs[0].value if score.outputs else None
    print(f"{score.task_id:<20} reward={reward}  status={score.status.value}")
for trial in result.trials:
    if trial.error is not None:
        print(f"{trial.id}: error={trial.error.type}: {trial.error.message}")

<a id="experiment"></a>
## 4. Create the Experiment and Evaluation

`publish_to_intake` references an Evaluation that already exists and never creates one — ingest
rejects a name the platform has not seen. So creating it is a caller-side step, done here.

Both calls pass `exist_ok=True`, which makes the cell safe to re-run.

In [ ]:
from nemo_platform import AsyncNeMoPlatform

client = AsyncNeMoPlatform(base_url=BASE_URL, max_retries=2)

experiment = await client.experiments.create(
    workspace=WORKSPACE, name="harbor-demo", description="Harbor -> Intake demo", exist_ok=True
)
evaluation = await client.evaluations.create(
    workspace=WORKSPACE,
    name=f"terminal-bench-2-1-{result.run_id}",
    experiment_ids=[experiment.id],
    dataset_name="terminal-bench-2-1",
    dataset_version="v1",
    exist_ok=True,
)

print(f"experiment : {experiment.name} ({experiment.id})")
print(f"evaluation : {evaluation.name}")

<a id="publish"></a>
## 5. Publish to Intake

For each trial, `publish_to_intake` posts the ATIF trajectory, resolves its root span, and writes
one evaluator-result row per metric output.

Two properties worth knowing:

- **Not atomic, but recoverable.** Every trial that can land does; failures are collected and
  raised together, carrying a partial report. The local result is the system of record, so you fix
  the problem and call it again — the run does not have to be repeated.
- **Idempotent.** The session id and every row id derive from the run and trial, not from the
  clock, so re-publishing replaces rows instead of duplicating them.

In [ ]:
from nemo_evaluator.intake.publish import publish_to_intake

report = await publish_to_intake(
    result,
    platform=client,
    experiment_id=evaluation.name,
    workspace=WORKSPACE,
    agent_name=AGENT,
    model_name=MODEL if AGENT not in KEYLESS_AGENTS else "none",
)

print(f"trials     : {report.trial_count}")
print(f"score rows : {report.evaluator_result_count}")
for omitted in report.skipped:
    print(f"skipped    : {omitted.trial_id}/{omitted.name} — {omitted.reason}")

<a id="studio"></a>
## 6. Open it in Studio

The same run, in the UI. These mirror two destinations Studio publishes in
`nmp.studio.studio_links` — `experiment_detail` for the Evaluation, `intake_session` for a single
trial's trajectory.

The URLs are printed in full so you can copy one straight into VS Code's Simple Browser
(**Simple Browser: Show** in the Command Palette) — clicking a link hands it to your system browser
instead. They resolve only while Studio is among the running services, which section 0 starts.

In [ ]:
from urllib.parse import quote

# Printed, not rendered as links: a Markdown link hides the URL, and VS Code hands clicks to the
# system browser — a plain URL is the one form you can paste into the integrated browser.
root = f"{BASE_URL.rstrip('/')}/studio/workspaces/{quote(WORKSPACE)}"
print("Evaluation:")
print(f"  {root}/experiment/harbor-demo/{quote(f'terminal-bench-2-1-{result.run_id}')}")
for published in report.published_trials:
    print(f"Trial {published.trial_id}:")
    print(f"  {root}/intake/sessions/{quote(published.session_id)}")

## Where to go next

- **Swap the tasks.** Any Hub task works: put its `org/task` id in `TASKS`. Browse them at
  [hub.harborframework.com](https://hub.harborframework.com/datasets). Heavier benchmarks carry much
  larger images, so try one outside a live demo first.
- **Swap the agent.** `AGENT` takes any Harbor agent — `claude-code`, `codex`, `mini-swe-agent`,
  `nemo-agent` — each with its own key and model conventions.
- **Compare runs.** Each run gets its own Evaluation under the `harbor-demo` Experiment. Pin
  the Evaluation name below to a fixed string when you *want* several runs gathered
  into one list, in place of the per-run `result.run_id`.

When you're finished, stop the platform. Its managed ClickHouse container stops with it, keeping
its data for the next run.